# Weapon Detection Training (Cloud Edition - V2.1 Persistent)

### 🛡️ PERSISTENCE MODE ENABLED
This version saves results **directly to your Google Drive** so you don't lose progress if the runtime stops.

### Instructions:
1. Enable **T4 GPU** (Runtime -> Change runtime type).
2. Mount your **Google Drive** in Cell 1.
3. Run all cells.

In [ ]:
!pip install ultralytics pyyaml
from google.colab import drive
import os
import yaml
import torch

# 1. Mount Drive immediately
drive.mount('/content/drive')

# 2. Create a persistent results folder
DRIVE_ROOT = '/content/drive/MyDrive/weapon_training_v2'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"✓ Results will be saved to: {DRIVE_ROOT}")

In [ ]:
zip_path = '/content/drive/MyDrive/weapon_dataset_colab.zip'

if os.path.exists(zip_path):
    print(f"✓ Found zip at {zip_path}. Unzipping to volatile /content/ for speed...")
    !unzip -q -o "{zip_path}" -d /content/
    
    yaml_path = '/content/data/unified_weapon_data.yaml'
    with open(yaml_path, 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['path'] = '/content/data/unified_yolo'
    with open(yaml_path, 'w') as f:
        yaml.dump(cfg, f)
    print("✓ Dataset ready!")
else:
    print("❌ ZIP NOT FOUND in Drive root. Please upload 'weapon_dataset_colab.zip' to your Google Drive FIRST.")

In [ ]:
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
DATA_YAML = "/content/data/unified_weapon_data.yaml"

COMMON = dict(
    data=DATA_YAML,
    epochs=100,
    patience=20,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cls=2.0, 
    mosaic=1.0,
    copy_paste=0.3,
    erasing=0.4, 
    save_period=10,
    project=DRIVE_ROOT, # CRITICAL: This points to Google Drive
    save=True,
    exist_ok=True,
    device=DEVICE
)

# We focus on Nano as requested
name, weights, imgsz, batch = "Normal_Compressed_v2", "yolov8n.pt", 640, 16

print(f"\nStarting Persistent Training: {name}")
model = YOLO(weights)
model.train(**COMMON, name=name, imgsz=imgsz, batch=batch)

print(f"\n✓ Training complete! Weights are safe in Google Drive: {DRIVE_ROOT}/{name}/weights/best.pt")